# Farm Risk Demo

This notebook walks through a simple agricultural finance workflow using `ag-finance-toolkit`.
It stays executable without external API access by using a realistic sample farm for the core metrics and Monte Carlo sections.


## 1. Define A Sample Farm

We start with a stylized Iowa corn and soybean operation. The assumptions below are intentionally explicit so the resulting metrics and risk outputs are easy to interpret.


In [ ]:
from agfin import calculate_all, simulate_farm_risk
from agfin.schemas import CropMixItem, FarmInputs

farm = FarmInputs(
    total_acres=500.0,
    crop_mix=[
        CropMixItem(crop="Corn", acres=300.0, expected_yield=190.0, expected_price=4.8),
        CropMixItem(crop="Soybeans", acres=200.0, expected_yield=55.0, expected_price=12.0),
    ],
    operating_costs=250_000.0,
    fixed_costs=80_000.0,
    debt_obligations=60_000.0,
    working_capital=100_000.0,
    current_liabilities=40_000.0,
    total_assets=1_500_000.0,
    total_liabilities=600_000.0,
    net_worth=900_000.0,
    insurance_income=5_000.0,
    govt_payments=10_000.0,
    interest_expense=25_000.0,
    depreciation=45_000.0,
    owner_draws=30_000.0,
)

farm


## 2. Calculate Core Financial Metrics

The package exposes a schema-aware `calculate_all(...)` helper that translates validated inputs into liquidity, solvency, profitability, and repayment outputs.


In [ ]:
metrics = calculate_all(farm)

{
    "working_capital": metrics.working_capital,
    "current_ratio": round(metrics.current_ratio, 2),
    "debt_to_asset": round(metrics.debt_to_asset, 2),
    "equity_ratio": round(metrics.equity_ratio, 2),
    "net_farm_income": round(metrics.net_farm_income, 2),
    "operating_margin": round(metrics.operating_margin, 3),
    "operating_expense_ratio": round(metrics.operating_expense_ratio, 3),
    "dscr": round(metrics.dscr, 2),
}


## 3. Simulate Income And Repayment Risk

We now layer uncertainty on top of the base farm assumptions. The Monte Carlo model applies multiplicative shocks to yield, price, and operating costs, then summarizes the resulting income and DSCR distributions.


In [ ]:
risk = simulate_farm_risk(
    farm,
    yield_cv=0.12,
    price_cv=0.15,
    cost_cv=0.08,
    runs=10_000,
    seed=42,
)

{
    "mean_income": round(risk.mean_income, 2),
    "p10_income": round(risk.p10_income, 2),
    "p50_income": round(risk.p50_income, 2),
    "p90_income": round(risk.p90_income, 2),
    "mean_dscr": round(risk.mean_dscr, 2),
    "p10_dscr": round(risk.p10_dscr, 2),
    "default_probability": round(risk.default_probability, 3),
    "worst_case_shortfall": round(risk.worst_case_shortfall, 2),
}


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

rng = np.random.default_rng(42)
yield_multiplier = np.clip(rng.normal(loc=1.0, scale=0.12, size=10_000), a_min=0.0, a_max=None)
price_multiplier = np.exp(rng.normal(loc=-0.5 * (0.15 ** 2), scale=0.15, size=10_000))
cost_multiplier = np.clip(rng.normal(loc=1.0, scale=0.08, size=10_000), a_min=0.0, a_max=None)

base_revenue = sum(item.acres * item.expected_yield * item.expected_price for item in farm.crop_mix)
income_draws = (
    base_revenue * yield_multiplier * price_multiplier
    - farm.operating_costs * cost_multiplier
    - farm.fixed_costs
    + farm.insurance_income
    + farm.govt_payments
)
dscr_draws = (
    income_draws + farm.depreciation + farm.interest_expense - farm.owner_draws
) / farm.debt_obligations

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(income_draws, bins=40, color="#4C7A34", alpha=0.85)
axes[0].axvline(risk.p10_income, color="#A63A2B", linestyle="--", label="P10 income")
axes[0].axvline(risk.p50_income, color="#1F4E79", linestyle="--", label="P50 income")
axes[0].set_title("Simulated Net Farm Income")
axes[0].set_xlabel("Income ($)")
axes[0].set_ylabel("Frequency")
axes[0].legend()

axes[1].hist(dscr_draws, bins=40, color="#C69214", alpha=0.85)
axes[1].axvline(1.0, color="#A63A2B", linestyle="--", label="DSCR = 1.0")
axes[1].axvline(risk.p10_dscr, color="#1F4E79", linestyle="--", label="P10 DSCR")
axes[1].set_title("Simulated DSCR")
axes[1].set_xlabel("DSCR")
axes[1].set_ylabel("Frequency")
axes[1].legend()

plt.tight_layout()


## 4. Interpreting The Output

A lender or operator would usually care about questions like:

- How large is the downside gap between average income and P10 income?
- How often does DSCR fall below 1.0?
- How severe is the worst simulated debt-service shortfall?

Those are exactly the summary fields returned by `simulate_farm_risk(...)`.


## 5. Optional Connector Examples

The package also includes public-data connectors for USDA NASS and NASA POWER. Those are useful for replacing hard-coded assumptions with external yield, price, or weather inputs, but they are left out of the executable flow here so this notebook can run cleanly without network access or API credentials.

Example imports:

```python
from agfin.connectors import get_crop_yield, get_crop_price, get_weather_history
```
